# MIMIC-IV and eICU — Learning and Holdout Set Generation

This notebook generates the **learning** and **holdout** CSV files from:
- The preprocessed **MIMIC-IV** dataset (used for centralized model training in MEDomics)
- The **eICU** per-hospital datasets (used for federated training and external validation)

Both datasets follow the MED3PA preprocessing protocol
(SAPS-II transformation + independent median imputation per set).

## Prerequisites
- Access to MIMIC-IV and eICU via PhysioNet:
  - https://physionet.org/content/mimiciv/
  - https://physionet.org/content/eicu-crd/
- Preprocessed input files generated by the MED3PA pipeline

## Output
For **MIMIC-IV**:
- `learning.csv` — 95% of data, used for centralized model training
- `holdout.csv`  — 5% of data, used for internal validation

For **eICU** (run once per hospital file):
- `Holdout_eicu_dataset_hospital_*.csv` — holdout set per hospital

> **Note:** Imputation is performed independently on each set to prevent data leakage.

## Configuration

Update the paths below to match your local environment.

**For MIMIC-IV:** set `INPUT_PATH` to your `MIMIC_saps.csv` file.

**For eICU:** set `INPUT_PATH` to a per-hospital file (e.g. `eicu_hospital_264.csv`)
and update `OUTPUT_LEARNING` / `OUTPUT_HOLDOUT` accordingly.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# ==============================
#  CONFIGURATION
#  Update paths to your local environment
# ==============================

# --- MIMIC-IV ---
INPUT_PATH      = "data/MIMIC_saps.csv"      # preprocessed MIMIC-IV (SAPS-II format)
OUTPUT_LEARNING = "data/learning.csv"
OUTPUT_HOLDOUT  = "data/holdout.csv"

# --- eICU (uncomment and update for each hospital) ---
# INPUT_PATH      = "data/eicu_hospital_264.csv"
# OUTPUT_LEARNING = "data/Learning_eicu_hospital_264.csv"
# OUTPUT_HOLDOUT  = "data/Holdout_eicu_dataset_hospital_264.csv"

TARGET = "deceased"

## 1. Load Data

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Target distribution:")
print(df[TARGET].value_counts(normalize=True).round(3))

## 2. Drop Unnecessary Columns

In [ ]:
# Drop identifier and metadata columns not needed for training
cols_to_drop = ["id", "anchor_year_group"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Remove unnamed and fully empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df = df.dropna(axis=1, how="all")

print(f"Shape after column removal: {df.shape}")

## 3. Stratified Train/Holdout Split

- **MIMIC-IV**: 95% learning / 5% holdout
- **eICU per hospital**: same split applied independently per hospital file

The split is stratified on the target column to preserve class proportions.

In [ ]:
# Stratified split to preserve target class proportions
learning, holdout = train_test_split(
    df,
    test_size=0.05,
    random_state=42,
    stratify=df[TARGET]
)

print(f"Learning set : {learning.shape}")
print(f"Holdout set  : {holdout.shape}")
print(f"Positive rate - Learning : {learning[TARGET].mean():.3f}")
print(f"Positive rate - Holdout  : {holdout[TARGET].mean():.3f}")

## 4. Independent Median Imputation

Imputation is performed independently on each set to prevent data leakage:
- The learning set uses its own median
- The holdout set uses its own median

This ensures no information from the holdout leaks into the training process,
and applies consistently for both MIMIC-IV and eICU splits.

In [ ]:
num_cols = learning.select_dtypes(include="number").columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

# Compute medians independently on each set
learning_medians = learning[num_cols].median()
holdout_medians  = holdout[num_cols].median()

learning[num_cols] = learning[num_cols].fillna(learning_medians)
holdout[num_cols]  = holdout[num_cols].fillna(holdout_medians)

# Export
learning.to_csv(OUTPUT_LEARNING, index=False)
holdout.to_csv(OUTPUT_HOLDOUT,   index=False)

print(f"Learning set saved : {learning.shape} -> {OUTPUT_LEARNING}")
print(f"Holdout set saved  : {holdout.shape}  -> {OUTPUT_HOLDOUT}")

✅ Learning : (15457, 16)  →  C:/Users/user/Downloads/learning.csv
✅ Holdout  : (814, 16)   →  C:/Users/user/Downloads/holdout.csv
